# Step 2 -- Propagation

In this notebook, we will propagate the annotations that we formatted in the previous notebook. To do this, we will use the HOGProp tool.

In [1]:
common_data_path = '/home/ubuntu/SIBBiodiversitySummerSchool2026Topic4/CommonData/Topic4CommonData'

## Running HOGPROP

HOGPROP requires three inputs:

- A GO ontology file (`go-basic.obo`)
- A GAF file containing extant annotations
- An OrthoXML file containing HOGs generated by FastOMA

The command below propagates GO annotations through the HOG hierarchy and stores the results in an HDF5 file.

We can run the hogprop tool with this command, however even with this small dataset this is quite slow.

In [ ]:
import subprocess
import os

cmd = [
    "hogprop",
    "--obo", os.path.join(common_data_path, 'Module4', 'go-basic.obo'),
    "--gaf", "../step1_annotation/extant_predictions.gaf.gz",
    "--go_filter", "all",
    "--combination_func", "max",
    "--oxml", f"{common_data_path}/Module2/fastoma_output/FastOMA_HOGs.orthoxml",
    "--results", "hogprop_results.h5"
]

print(" ".join(cmd))
subprocess.run(cmd)

hogprop --obo /home/ubuntu/SIBBiodiversitySummerSchool2026Topic4/CommonData/Topic4CommonData/Module4/go-basic.obo --gaf ../step1_annotation/extant_predictions.gaf.gz --go_filter all --combination_func max --oxml /home/ubuntu/SIBBiodiversitySummerSchool2026Topic4/CommonData/Topic4CommonData/Module2/fastoma_output/FastOMA_HOGs.orthoxml --results hogprop_results.h5


Loading species: 10 species [00:07,  1.35 species/s]
Loading Annotations: 20321359it [01:15, 267737.30it/s]


time to read annots: 77.275253


Propagating through HOGs: 1865it [36:11,  1.32s/it]

---

The example above is suitable for small datasets and for understanding how HOGPROP works. For larger datasets, HogProp is typically run on a compute cluster using a job scheduler such as SLURM.

The script to run HogProp on all the proteomes in parallel on an HPC is `sbatch hogprop.sh` in this directory.

Then, once all jobs are complete we can run `source hogprop-convert.sh` in order to convert to a single output database. This can then be used in the next step.

## Questions

For larger datasets, HOGPROP is usually submitted to a cluster.

Open `hogprop.sh` and `hogprop-convert.sh` and identify:

1. Which file contains the GO ontology?
2. Which file contains the GAF annotations?
3. Which file contains the HOG structure?
4. Where are the results written?
5. How many HOGs have annotations?

1. Which file contains the GO ontology?

`go-basic.obo`

2. Which file contains the GAF annotations?

`extant_predictions.gaf.gz`

3. Which file contains the HOG structure?

`FastOMA_HOGs.orthoxml`

4. Where are the *final* results written?

`hogprop_output.h5`

5. How many HOGs have annotations?

In [13]:
#How many HOG annotation records were generated?

import tables
import os

h5 = tables.open_file(os.path.join(common_data_path, 'Module4', 'precomputed_results', "hogprop_output.h5"))

print(
    h5.root.HOGAnnotations.GeneOntology.nrows,
    "HOG annotation records"
)

h5.close()

11799967 HOG annotation records


In [12]:
#How many unique HOGs received at least one GO annotation?

import numpy as np

h5 = tables.open_file(os.path.join(common_data_path, 'Module4', 'precomputed_results', "hogprop_output.h5"))

table = h5.root.HOGAnnotations.GeneOntology

hog_ids = table.col("HogNr")

print("Unique HOGs:", len(np.unique(hog_ids)))

h5.close()

Unique HOGs: 100872


In [ ]:
#parallelized version
# ran with --no_covert parameter, then converted later with hogprop-convert.sh

h5 = tables.open_file(os.path.join(common_data_path, 'Module4', 'precomputed_results', "hogprop_output.h5"))

print(h5)

/home/ubuntu/SIBBiodiversitySummerSchool2026Topic4/CommonData/Topic4CommonData/Module4/precomputed_results/hogprop_output.h5 (File) ''
Last modif.: '2026-06-15T09:38:26+00:00'
Object Tree: 
/ (RootGroup) ''
/HogLevel (Table(117633,)fletcher32, shuffle, zlib(6)) 'similar to /HogLevel in pyomadb'
/Annotations (Group) ''
/Annotations/GeneOntology_HOGPROP (Table(17566826,)fletcher32, shuffle, zlib(6)) 'HOGPROP gene ontology extant predictions'
/HOGAnnotations (Group) ''
/HOGAnnotations/GeneOntology (Table(11799967,)fletcher32, shuffle, zlib(6)) 'HOGPROP gene ontology ancestral predictions'



In [ ]:
#non-parallelized version
#ran with NO --no_covert parameter, NOT converted later with hogprop-convert.sh

h5 = tables.open_file(os.path.join('..', 'step2_propagation',  "hogprop_results.h5"))

print(h5)

../step2_propagation/hogprop_results.h5 (File) ''
Last modif.: '2026-06-15T15:56:48+00:00'
Object Tree: 
/ (RootGroup) ''
/HOG:0000001 (Group) ''
/HOG:0000001/id_types (CArray(15,)fletcher32, zlib(6)) ''
/HOG:0000001/ids (CArray(15,)fletcher32, zlib(6)) ''
/HOG:0000001/scores (CArray(15, 14)fletcher32, zlib(6)) ''
/HOG:0000001/terms (CArray(14,)fletcher32, zlib(6)) ''
/HOG:0000002 (Group) ''
/HOG:0000002/id_types (CArray(13,)fletcher32, zlib(6)) ''
/HOG:0000002/ids (CArray(13,)fletcher32, zlib(6)) ''
/HOG:0000002/scores (CArray(13, 19)fletcher32, zlib(6)) ''
/HOG:0000002/terms (CArray(19,)fletcher32, zlib(6)) ''
/HOG:0000003 (Group) ''
/HOG:0000003/id_types (CArray(19,)fletcher32, zlib(6)) ''
/HOG:0000003/ids (CArray(19,)fletcher32, zlib(6)) ''
/HOG:0000003/scores (CArray(19, 28)fletcher32, zlib(6)) ''
/HOG:0000003/terms (CArray(28,)fletcher32, zlib(6)) ''
/HOG:0000004 (Group) ''
/HOG:0000004/id_types (CArray(19,)fletcher32, zlib(6)) ''
/HOG:0000004/ids (CArray(19,)fletcher32, zlib(6))